In [2]:
!pip install pretty_midi
import os
import pretty_midi
import numpy as np
import torch



     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 99.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 4.4 MB/s eta 0:00:00
  Created wheel for pretty_midi: filename=pretty_midi-0.2.11-py3-none-any.whl size=5595886 sha256=2f8f1f63f6ca60b225c1a5bc75c6fccf0ae3291f39443ba6dd066b5bfa51bae1
  Stored in directory: /root/.cache/pip/wheels/f4/ad/93/a7042fe12668827574927ade9deec7f29aad2a1001b1501882
Successfully built pretty_midi


In [3]:
# ============================================================
# 1. 구글 드라이브 마운트 및 파일 압축 해제
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

# 🚨 구글 드라이브에 있는 ZIP 파일 경로와 Colab에 압축을 풀 로컬 경로 설정
ZIP_PATH = "/content/drive/MyDrive/ML/midivae/categorized_midi.zip"
UNZIP_DIR = "/content/categorized_midi"

# Colab 로컬 환경에 압축 해제 (!unzip 명령어가 파이썬 내장 모듈보다 훨씬 빠릅니다)
if not os.path.exists(UNZIP_DIR):
    print(f"📦 압축 해제 시작: {ZIP_PATH} -> {UNZIP_DIR}")
    !unzip -q "{ZIP_PATH}" -d "{UNZIP_DIR}"
    print("✅ 압축 해제 완료!")
else:
    print("✅ 이미 압축이 해제되어 있습니다.")

# ============================================================
# 2. 경로 및 설정
# ============================================================
# 🚨 압축이 해제된 Colab 로컬 경로를 원본 데이터 경로로 지정
RAW_DATA_ROOT = "/content/categorized_midi/categorized_midi"

# 🚨 Train 데이터로더가 바라볼 최종 텐서(.pt) 저장 경로 (결과는 안전하게 드라이브에 저장)
SAVE_DATA_ROOT = "/content/drive/MyDrive/ML/midivae/data"

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
📦 압축 해제 시작: /content/drive/MyDrive/ML/midivae/categorized_midi.zip -> /content/categorized_midi
✅ 압축 해제 완료!


In [4]:
RAW_DATA_ROOT = "/content/categorized_midi/categorized_midi"

In [7]:
# ============================================================
# 1. 경로 및 설정
# ============================================================
# 🚨 카테고리별로 분류된 원본 MIDI 폴더 경로
#RAW_DATA_ROOT = "/content/drive/MyDrive/ML/MIDIVAE/categorized_midi"

# 🚨 Train 데이터로더가 바라볼 최종 텐서(.pt) 저장 경로
SAVE_DATA_ROOT = "/content/drive/MyDrive/ML/midivae/data"

MOODS = [
    'angry', 'exciting', 'fear', 'funny', 'happy',
    'lazy', 'magnificent', 'quiet', 'romantic', 'sad', 'warm'
]

# 훈련 시퀀스 길이가 8이므로, Next-Step 예측을 위해선 최소 9개의 청크가 필요합니다.
MIN_CHUNKS_REQUIRED = 9

# 🚨 각 감정별로 전처리하여 저장할 최대 파일 개수
MAX_FILES_PER_MOOD = 1000

# ============================================================
# 2. 전처리 함수 정의
# ============================================================
def process_midi_dataset():
    total_success = 0
    total_skipped = 0

    print(f"🚀 전처리 시작: {RAW_DATA_ROOT} -> {SAVE_DATA_ROOT}")
    print(f"🎯 각 감정별 목표 최대 처리 개수: {MAX_FILES_PER_MOOD}개\n")

    for mood in MOODS:
        raw_mood_dir = os.path.join(RAW_DATA_ROOT, mood)
        save_mood_dir = os.path.join(SAVE_DATA_ROOT, mood)

        # 저장할 감정별 하위 폴더 생성
        os.makedirs(save_mood_dir, exist_ok=True)

        if not os.path.exists(raw_mood_dir):
            print(f"⚠️ {mood} 폴더를 찾을 수 없어 건너뜁니다.")
            continue

        midi_files = [f for f in os.listdir(raw_mood_dir) if f.lower().endswith(('.mid', '.midi'))]
        print(f"🎧 [{mood.upper()}] 폴더 처리 중... (탐색 대상 파일 {len(midi_files)}개)")

        success_count = 0
        skipped_count = 0

        for filename in midi_files:
            # 🚨 10,000개를 성공적으로 저장했다면, 현재 감정 루프 종료
            if success_count >= MAX_FILES_PER_MOOD:
                print(f"   ✅ 목표치 {MAX_FILES_PER_MOOD}개 달성! 다음 감정으로 넘어갑니다.")
                break

            filepath = os.path.join(raw_mood_dir, filename)
            name_no_ext = os.path.splitext(filename)[0]

            try:
                pm = pretty_midi.PrettyMIDI(filepath)

                # 4/4 박자 필터링
                if not pm.time_signature_changes or not (pm.time_signature_changes[0].numerator == 4 and pm.time_signature_changes[0].denominator == 4):
                    skipped_count += 1
                    continue

                # BPM 및 스텝 계산
                bpm = pm.estimate_tempo()
                if bpm <= 0:
                    skipped_count += 1
                    continue

                # 16 position/bar 기준 (1마디 = 16스텝, 2마디 = 32스텝)
                step_duration = (60.0 / bpm) / 4.0
                total_steps = int(pm.get_end_time() / step_duration)
                chunk_steps = 32

                if total_steps < chunk_steps:
                    skipped_count += 1
                    continue

                # 4채널 피아노 롤 추출 [채널, 88건반, 시간]
                piano_roll = np.zeros((4, 88, total_steps), dtype=np.uint8)
                for inst in pm.instruments:
                    # 악기 분류 (0: 멜로디, 1: 하모니, 2: 베이스, 3: 드럼)
                    c = 3 if inst.is_drum else (2 if 32 <= inst.program <= 39 else (1 if (40 <= inst.program <= 55) or (88 <= inst.program <= 95) else 0))

                    for note in inst.notes:
                        if 21 <= note.pitch <= 108:
                            s = int(note.start / step_duration)
                            e = max(s + 1, int(note.end / step_duration))
                            e = min(e, total_steps)
                            if s < total_steps:
                                piano_roll[c, note.pitch - 21, s:e] = 1

                # 2마디(32스텝) 단위 청킹
                chunks = []
                for start in range(0, total_steps - chunk_steps + 1, chunk_steps):
                    end = start + chunk_steps
                    chunks.append(piano_roll[:, :, start:end])

                # 시퀀스 길이 방어 로직: 충분한 길이의 곡만 저장
                if len(chunks) >= MIN_CHUNKS_REQUIRED:
                    save_path = os.path.join(save_mood_dir, f"{name_no_ext}.pt")
                    torch.save({"chunks": torch.tensor(np.array(chunks), dtype=torch.uint8)}, save_path)
                    success_count += 1
                else:
                    skipped_count += 1

            except Exception:
                skipped_count += 1

        print(f"   -> 완료: 저장 {success_count}곡 | 스킵 {skipped_count}곡")
        total_success += success_count
        total_skipped += skipped_count

    print("============================================================")
    print(f"🎉 전체 전처리 완료! 완벽하게 가공된 총 {total_success}곡이 준비되었습니다.")
    print("============================================================")


process_midi_dataset()


🚀 전처리 시작: /content/categorized_midi/categorized_midi -> /content/drive/MyDrive/ML/midivae/data
🎯 각 감정별 목표 최대 처리 개수: 1000개

🎧 [ANGRY] 폴더 처리 중... (탐색 대상 파일 8739개)
   ✅ 목표치 1000개 달성! 다음 감정으로 넘어갑니다.
   -> 완료: 저장 1000곡 | 스킵 0곡
🎧 [EXCITING] 폴더 처리 중... (탐색 대상 파일 20948개)
   ✅ 목표치 1000개 달성! 다음 감정으로 넘어갑니다.
   -> 완료: 저장 1000곡 | 스킵 0곡
🎧 [FEAR] 폴더 처리 중... (탐색 대상 파일 3621개)
   ✅ 목표치 1000개 달성! 다음 감정으로 넘어갑니다.
   -> 완료: 저장 1000곡 | 스킵 4곡
🎧 [FUNNY] 폴더 처리 중... (탐색 대상 파일 12565개)
   ✅ 목표치 1000개 달성! 다음 감정으로 넘어갑니다.
   -> 완료: 저장 1000곡 | 스킵 0곡
🎧 [HAPPY] 폴더 처리 중... (탐색 대상 파일 13291개)
   ✅ 목표치 1000개 달성! 다음 감정으로 넘어갑니다.
   -> 완료: 저장 1000곡 | 스킵 0곡
🎧 [LAZY] 폴더 처리 중... (탐색 대상 파일 4622개)
   ✅ 목표치 1000개 달성! 다음 감정으로 넘어갑니다.
   -> 완료: 저장 1000곡 | 스킵 0곡
🎧 [MAGNIFICENT] 폴더 처리 중... (탐색 대상 파일 2792개)
   ✅ 목표치 1000개 달성! 다음 감정으로 넘어갑니다.
   -> 완료: 저장 1000곡 | 스킵 0곡
🎧 [QUIET] 폴더 처리 중... (탐색 대상 파일 4431개)
   ✅ 목표치 1000개 달성! 다음 감정으로 넘어갑니다.
   -> 완료: 저장 1000곡 | 스킵 1곡
🎧 [ROMANTIC] 폴더 처리 중... (탐색 대상 파일 12886개)
   ✅ 목표치 1000개 달성! 다음 감정으로 넘어갑니다.